# 04 | Renda pessoal por habitante (BEA)

Preparamos a renda anual por pessoa para usar como informação econômica na comparação. O notebook aceita a exportação bruta SAINC1 ou, nesta máquina, o CSV processado anteriormente.

## Bloco 1: Abrir

Se o bruto existir, lemos `data/raw/bea/personal_income.csv`. Caso contrário, usamos `data/processed/bea_income_2019_2024.csv` e avisamos na tela.

In [4]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
bruto = ROOT / "data/raw/bea/personal_income.csv"
processado = ROOT / "data/processed/bea_income_2019_2024.csv"
if bruto.exists():
    bea = pd.read_csv(bruto, skiprows=3, encoding="utf-8-sig", dtype={"GeoFIPS": str}, low_memory=False)
    origem = "bruto"
elif processado.exists():
    bea = pd.read_csv(processado)
    origem = "processado"
else:
    raise FileNotFoundError("Exporte a tabela SAINC1 do BEA e coloque o CSV em data/raw/bea/personal_income.csv.")
print("Origem:", origem, "| Linhas lidas:", len(bea))

Origem: bruto | Linhas lidas: 66


## Bloco 2: Organizar

A exportação bruta tem anos como colunas; o CSV processado já tem uma linha por estado e ano.

In [5]:
if origem == "bruto":
    anos = [str(a) for a in range(2019, 2025)]
    longa = bea[["GeoFIPS", "GeoName"] + anos].melt(id_vars=["GeoFIPS", "GeoName"], value_vars=anos, var_name="year", value_name="income_per_capita")
    longa = longa.rename(columns={"GeoName": "state"})
    longa["state"] = longa["state"].str.replace(r"\s*\*$", "", regex=True).str.strip()
else:
    longa = bea.copy()
longa["year"] = pd.to_numeric(longa["year"], errors="coerce")
longa["income_per_capita"] = pd.to_numeric(longa["income_per_capita"], errors="coerce")
cdc = pd.read_csv(ROOT / "data/raw/cdc_homicides_2019_2024.csv")
estados = set(cdc["name"]) - {"District of Columbia"}
limpo = longa[longa["state"].isin(estados) & longa["year"].between(2019, 2024)][["state", "year", "income_per_capita"]].copy()
limpo["year"] = limpo["year"].astype(int)
display(limpo.head())

,state,year,income_per_capita
1,Alabama,2019,42996.0
2,Alaska,2019,60483.0
3,Arizona,2019,47646.0
4,Arkansas,2019,43745.0
5,California,2019,64230.0


## Bloco 3: Validar e salvar

Esperamos 300 estado-anos, sem valores de renda vazios ou não positivos. A renda é nominal, sem ajuste pela inflação.

In [6]:
assert limpo.shape == (300, 3)
assert not limpo.duplicated(["state", "year"]).any()
assert limpo["income_per_capita"].notna().all()
assert (limpo["income_per_capita"] > 0).all()
processado.parent.mkdir(parents=True, exist_ok=True)
limpo.to_csv(processado, index=False)
print("OK, salvo:", processado.relative_to(ROOT))

OK, salvo: data/processed/bea_income_2019_2024.csv
